# 4. Statistical Tests

Hypothesis tests and effect sizes for the associations observed in notebook 3 (training period 2018–2021). Mann–Whitney and Kruskal–Wallis tests use a random subsample of 500,000 flights; chi-square tests and bootstrap intervals use all training flights. Results are written to `EDA/data/paper_tables/`.

With millions of flights almost every difference is statistically significant, so **effect sizes** (Cohen's *d*, Cramér's *V*, ε²) are the main quantities to interpret.

## 4.1 Setup and data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists())  # project root
DATA = ROOT / 'data' / 'processed'          # outputs of 02_weather_processing and 03_flights_information_settings
TRAIN_END = '2022-01-01'

FLIGHT_COLS = ['row_id', 'FlightDate', 'Airline', 'Origin', 'Dest', 'OriginState', 'DestState',
               'CRSDepHour', 'Month', 'DayOfWeek', 'Distance', 'CRSElapsedTime',
               'ArrDelay', 'ArrDelayMinutes', 'ArrDel15']
WX = ['temp_f', 'dewpt_f', 'humidity', 'wind_mph', 'gust_mph', 'visibility_mi', 'precip_in', 'pressure',
      'ceiling_ft', 'has_rain', 'has_snow', 'has_fog', 'has_mist', 'has_thunder', 'has_freezing',
      'is_heavy', 'has_gust', 'flight_cat_ord']
WX_COLS = ['row_id'] + [f'{p}_{c}' for p in ('org', 'dst') for c in WX]

fl = pd.read_parquet(DATA / 'flights_clean.parquet', columns=FLIGHT_COLS)
wx = pd.read_parquet(DATA / 'wx_oracle.parquet', columns=WX_COLS)   # realized weather at scheduled times (descriptive use)
assert (fl['row_id'].values == wx['row_id'].values).all()
df_all = pd.concat([fl, wx.drop(columns='row_id')], axis=1)
del fl, wx
df_all['FlightDate'] = pd.to_datetime(df_all['FlightDate'])
FLIGHT_CAT = {0: 'VFR', 1: 'MVFR', 2: 'IFR', 3: 'LIFR'}            # FAA category (ceiling and visibility)
for p in ('org', 'dst'):
    df_all[f'{p}_ceiling_cat'] = df_all[f'{p}_flight_cat_ord'].map(FLIGHT_CAT)

df = df_all[df_all['FlightDate'] < TRAIN_END].reset_index(drop=True)   # training period
print(f'All flights: {len(df_all):,} | training period (2018-2021): {len(df):,}')

STATS_DIR = ROOT / 'EDA' / 'data' / 'paper_tables'
STATS_DIR.mkdir(parents=True, exist_ok=True)

# Mann-Whitney and Kruskal-Wallis on a subsample; chi-square on all training flights
SAMPLE_SIZE = 500_000
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42)
print(f'Sample for slow tests: {len(df_sample):,}')

All flights: 9,981,331 | training period (2018-2021): 8,550,324
Sample for slow tests: 500,000


In [2]:
# Helper functions
def cohens_d(x, y):
    """Cohen's d for effect size between two groups."""
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled_std = np.sqrt(((nx-1)*vx + (ny-1)*vy) / (nx+ny-2))
    if pooled_std == 0:
        return np.nan
    return (x.mean() - y.mean()) / pooled_std

def cramers_v(chi2, n, r, c):
    """Cramér's V for effect size in contingency table."""
    return np.sqrt(chi2 / (n * min(r-1, c-1)))

def interpret_d(d):
    """Cohen's d interpretation."""
    abs_d = abs(d)
    if abs_d < 0.2: return "negligible"
    elif abs_d < 0.5: return "small"
    elif abs_d < 0.8: return "medium"
    else: return "large"

def interpret_v(v):
    """Cramér's V interpretation."""
    if v < 0.1: return "negligible"
    elif v < 0.3: return "small"
    elif v < 0.5: return "medium"
    else: return "large"

def format_p(p):
    """Format p-value for reporting."""
    if p < 0.001: return "< 0.001"
    elif p < 0.01: return f"< 0.01 (p={p:.3f})"
    else: return f"{p:.3f}"

print('Helpers defined')

Helpers defined


## 4.2 Weather conditions at the origin

For each condition: $H_0$ — the delay distribution is the same with and without the condition; $H_1$ — delays are larger with the condition (one-sided Mann–Whitney *U*). Cohen's *d* measures the difference in mean delay; Cramér's *V* (chi-square on `ArrDel15`) measures the association with the delay indicator.

In [3]:
flags = ['org_has_rain', 'org_has_snow', 'org_has_fog', 'org_has_mist', 'org_has_thunder', 'org_has_freezing']
flag_labels = ['Rain', 'Snow', 'Fog', 'Mist', 'Thunder', 'Freezing']

results = []
for flag, label in zip(flags, flag_labels):
    no_flag = df_sample.loc[df_sample[flag] == 0, 'ArrDelayMinutes'].values
    with_flag = df_sample.loc[df_sample[flag] == 1, 'ArrDelayMinutes'].values
    
    if len(with_flag) < 30:
        continue
    
    # Mann-Whitney U test (one-sided: with_flag > no_flag)
    stat, p = stats.mannwhitneyu(with_flag, no_flag, alternative='greater')
    
    # Cohen's d
    d = cohens_d(pd.Series(with_flag), pd.Series(no_flag))
    
    # Chi-square on ArrDel15 (all training flights)
    ct = pd.crosstab(df[flag], df['ArrDel15'])
    chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
    n_total = ct.sum().sum()
    v = cramers_v(chi2, n_total, ct.shape[0], ct.shape[1])
    
    results.append({
        'Condition': label,
        'N (with)': (df[flag] == 1).sum(),
        'Mean delay w/o': no_flag.mean(),
        'Mean delay w/': with_flag.mean(),
        'Δ mean delay': with_flag.mean() - no_flag.mean(),
        'MWU p-value': format_p(p),
        "Cohen's d": round(d, 3),
        'Effect (d)': interpret_d(d),
        'Chi² p-value': format_p(p_chi),
        "Cramér's V": round(v, 3),
        'Effect (V)': interpret_v(v),
    })

weather_test_df = pd.DataFrame(results)
weather_test_df.to_csv(STATS_DIR / 'stats_weather_flags.csv', index=False)
weather_test_df

,Condition,N (with),Mean delay w/o,Mean delay w/,Δ mean delay,MWU p-value,Cohen's d,Effect (d),Chi² p-value,Cramér's V,Effect (V)
0,Rain,588874,11.6543,23.3958,11.7415,< 0.001,0.2710,small,< 0.001,0.0870,negligible
1,Snow,74764,12.2128,40.3375,28.1247,< 0.001,0.6490,medium,< 0.001,0.0820,negligible
2,Fog,76334,12.3538,24.0974,11.7436,< 0.001,0.2700,small,< 0.001,0.0200,negligible
3,Mist,531924,11.9092,20.7453,8.8361,< 0.001,0.2040,small,< 0.001,0.0650,negligible
4,Thunder,93418,12.0205,52.9811,40.9606,< 0.001,0.9470,large,< 0.001,0.1070,small
5,Freezing,10715,12.4084,52.5321,40.1237,< 0.001,0.9240,large,< 0.001,0.0390,negligible


## 4.3 FAA flight category
Kruskal–Wallis test across VFR, MVFR, IFR and LIFR, followed by pairwise one-sided Mann–Whitney tests with Bonferroni correction.

In [4]:
cat_order = ['VFR', 'MVFR', 'IFR', 'LIFR']
groups = [df_sample.loc[df_sample['org_ceiling_cat'] == c, 'ArrDelayMinutes'].values
          for c in cat_order]

# Kruskal-Wallis omnibus test
H_stat, p_kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis H-statistic: {H_stat:.2f}')
print(f'Kruskal-Wallis p-value: {format_p(p_kw)}')

# Epsilon-squared effect size
n_total = sum(len(g) for g in groups)
k = len(groups)
eps_sq = (H_stat - k + 1) / (n_total - k)
print(f'ε² (epsilon-squared): {eps_sq:.4f}')
eps_interp = 'negligible' if eps_sq < 0.01 else ('small' if eps_sq < 0.06 else ('medium' if eps_sq < 0.14 else 'large'))
print(f'Effect size (ε²): {eps_interp}')

Kruskal-Wallis H-statistic: 2981.56
Kruskal-Wallis p-value: < 0.001
ε² (epsilon-squared): 0.0060
Effect size (ε²): negligible


In [5]:
posthoc = []
for c1, c2 in combinations(cat_order, 2):
    g1 = df_sample.loc[df_sample['org_ceiling_cat'] == c1, 'ArrDelayMinutes']
    g2 = df_sample.loc[df_sample['org_ceiling_cat'] == c2, 'ArrDelayMinutes']
    stat, p = stats.mannwhitneyu(g2, g1, alternative='greater')
    d = cohens_d(g2, g1)
    # Bonferroni correction (6 comparisons)
    p_adj = min(p * 6, 1.0)
    posthoc.append({
        'Comparison': f'{c2} vs {c1}',
        'Mean delay diff': g2.mean() - g1.mean(),
        'MWU p (adj)': format_p(p_adj),
        "Cohen's d": round(d, 3),
        'Effect': interpret_d(d),
    })

posthoc_df = pd.DataFrame(posthoc)
posthoc_df.to_csv(STATS_DIR / 'stats_ceiling_posthoc.csv', index=False)
posthoc_df

,Comparison,Mean delay diff,MWU p (adj),Cohen's d,Effect
0,MVFR vs VFR,2.1274,< 0.001,0.0510,negligible
1,IFR vs VFR,7.4510,< 0.001,0.1760,negligible
2,LIFR vs VFR,11.2119,< 0.001,0.2640,small
3,IFR vs MVFR,5.3236,< 0.001,0.1120,negligible
4,LIFR vs MVFR,9.0844,< 0.001,0.1870,negligible
5,LIFR vs IFR,3.7609,< 0.001,0.0640,negligible


## 4.4 Origin states
Kruskal–Wallis test across the ten states and pairwise effect sizes.

In [6]:
states = sorted(df['OriginState'].unique())
groups_state = [df_sample.loc[df_sample['OriginState'] == s, 'ArrDelayMinutes'].values
                for s in states]

H_state, p_state = stats.kruskal(*groups_state)
print(f'Kruskal-Wallis across {len(states)} states:')
print(f'  H-statistic: {H_state:.2f}')
print(f'  p-value: {format_p(p_state)}')

n_total = sum(len(g) for g in groups_state)
k = len(groups_state)
eps_sq_state = (H_state - k + 1) / (n_total - k)
print(f'  ε²: {eps_sq_state:.4f}')

# Per-state delay rate + CI
state_stats = df.groupby('OriginState').agg(
    n_flights=('ArrDelayMinutes', 'size'),
    mean_delay=('ArrDelayMinutes', 'mean'),
    pct_delayed=('ArrDel15', 'mean'),
)
state_stats['ci_lower'] = state_stats['pct_delayed'] - 1.96 * np.sqrt(state_stats['pct_delayed']*(1-state_stats['pct_delayed'])/state_stats['n_flights'])
state_stats['ci_upper'] = state_stats['pct_delayed'] + 1.96 * np.sqrt(state_stats['pct_delayed']*(1-state_stats['pct_delayed'])/state_stats['n_flights'])
state_stats = state_stats.sort_values('pct_delayed', ascending=False)
state_stats

Kruskal-Wallis across 10 states:
  H-statistic: 1015.66
  p-value: < 0.001
  ε²: 0.0020


,n_flights,mean_delay,pct_delayed,ci_lower,ci_upper
OriginState,,,,,
IL,665170,14.7005,0.2073,0.2063,0.2083
CO,571514,14.6340,0.2018,0.2008,0.2029
NY,692266,14.7116,0.1865,0.1856,0.1875
TX,1558038,13.0303,0.1862,0.1856,0.1868
FL,1123517,12.7879,0.1719,0.1712,0.1726
VA,541482,13.3526,0.1676,0.1666,0.1686
NC,626991,11.8540,0.1673,0.1663,0.1682
CA,1652902,10.8178,0.1584,0.1578,0.1589
WA,423042,9.6322,0.1539,0.1528,0.1550


In [7]:
state_pairs = []
for s1, s2 in combinations(states, 2):
    g1 = df_sample.loc[df_sample['OriginState'] == s1, 'ArrDelayMinutes']
    g2 = df_sample.loc[df_sample['OriginState'] == s2, 'ArrDelayMinutes']
    d = cohens_d(g1, g2)
    state_pairs.append({'pair': f'{s1} vs {s2}', 'd': round(abs(d), 3), 
                        'diff_mean': g1.mean() - g2.mean()})

state_pair_df = pd.DataFrame(state_pairs).sort_values('d', ascending=False)
print("Top 5 most different state pairs:")
print(state_pair_df.head())
print("\nBottom 5 (most similar) state pairs:")
print(state_pair_df.tail())
state_pair_df.to_csv(STATS_DIR / 'stats_state_pairs.csv', index=False)

Top 5 most different state pairs:
        pair      d  diff_mean
24  GA vs IL 0.1140    -4.8328
34  IL vs WA 0.1140     4.8070
10  CO vs GA 0.1130     5.0649
16  CO vs WA 0.1110     5.0391
26  GA vs NY 0.1110    -4.7993

Bottom 5 (most similar) state pairs:
        pair      d  diff_mean
21  FL vs TX 0.0070    -0.3045
11  CO vs IL 0.0050     0.2322
13  CO vs NY 0.0050     0.2656
29  GA vs WA 0.0010    -0.0258
31  IL vs NY 0.0010     0.0334


## 4.5 Bootstrap confidence intervals
Percentile bootstrap (300 resamples) of the delay rate under each weather condition.

In [8]:
def bootstrap_ci(data, statistic=np.mean, n_boot=1000, alpha=0.05, random_state=42):
    """Bootstrap 95% CI for a statistic."""
    rng = np.random.RandomState(random_state)
    n = len(data)
    boots = np.array([statistic(rng.choice(data, size=n, replace=True)) for _ in range(n_boot)])
    lower = np.percentile(boots, 100 * alpha / 2)
    upper = np.percentile(boots, 100 * (1 - alpha / 2))
    return lower, upper

# Bootstrap CI of the delay rate under each weather condition (all training flights)
boot_results = []
for flag, label in zip(flags, flag_labels):
    with_flag = df.loc[df[flag] == 1, 'ArrDel15'].values
    if len(with_flag) < 100:
        continue
    rate = with_flag.mean()
    # 300 resamples suffice for the large group sizes
    lo, hi = bootstrap_ci(with_flag, n_boot=300)
    boot_results.append({
        'Condition': label,
        'N': len(with_flag),
        'Delay rate': round(rate, 4),
        'CI lower (95%)': round(lo, 4),
        'CI upper (95%)': round(hi, 4),
        'CI width (pp)': round((hi - lo) * 100, 2),
    })

boot_df = pd.DataFrame(boot_results)
boot_df.to_csv(STATS_DIR / 'stats_bootstrap_ci.csv', index=False)
boot_df

,Condition,N,Delay rate,CI lower (95%),CI upper (95%),CI width (pp)
0,Rain,588874,0.2959,0.2947,0.2971,0.2400
1,Snow,74764,0.5060,0.5023,0.5094,0.7200
2,Fog,76334,0.2556,0.2528,0.2589,0.6100
3,Mist,531924,0.2697,0.2685,0.2709,0.2300
4,Thunder,93418,0.5595,0.5560,0.5621,0.6100
5,Freezing,10715,0.5939,0.5840,0.6037,1.9700


## 4.6 Summary

In [9]:
# Print all key findings
print('=' * 70)
print('KEY STATISTICAL FINDINGS')
print('=' * 70)
print()
print('1. Weather-condition effect sizes:')
for _, row in weather_test_df.iterrows():
    print(f"""  {row['Condition']:10} | d = {row["Cohen's d"]:+.3f} ({row['Effect (d)']:10}) | Cramér V = {row["Cramér's V"]:.3f}""")
print()
print(f'2. FAA flight category (Kruskal-Wallis): p = {format_p(p_kw)}, eps^2 = {eps_sq:.4f}')
print()
print(f'3. Origin states (Kruskal-Wallis): p = {format_p(p_state)}, eps^2 = {eps_sq_state:.4f}')
print(f'   → {"Reject H0" if p_state < 0.05 else "Fail to reject H0"}: states DO have different delay distributions')

KEY STATISTICAL FINDINGS

1. Weather-condition effect sizes:
  Rain       | d = +0.271 (small     ) | Cramér V = 0.087
  Snow       | d = +0.649 (medium    ) | Cramér V = 0.082
  Fog        | d = +0.270 (small     ) | Cramér V = 0.020
  Mist       | d = +0.204 (small     ) | Cramér V = 0.065
  Thunder    | d = +0.947 (large     ) | Cramér V = 0.107
  Freezing   | d = +0.924 (large     ) | Cramér V = 0.039

2. FAA flight category (Kruskal-Wallis): p = < 0.001, eps^2 = 0.0060

3. Origin states (Kruskal-Wallis): p = < 0.001, eps^2 = 0.0020
   → Reject H0: states DO have different delay distributions


In [10]:
summary = weather_test_df[['Condition', 'N (with)', 'Δ mean delay', 'MWU p-value', "Cohen's d", 'Effect (d)']].copy()
summary.columns = ['Weather Condition', 'N (with)', 'Δ Mean Delay (min)', 'MWU p-value', 'Cohen\'s d', 'Effect Size']
summary.to_csv(STATS_DIR / 'table_stats_summary.csv', index=False)
print('Summary table:')
print(summary.to_string(index=False))

Summary table:
Weather Condition  N (with)  Δ Mean Delay (min) MWU p-value  Cohen's d Effect Size
             Rain    588874             11.7415     < 0.001     0.2710       small
             Snow     74764             28.1247     < 0.001     0.6490      medium
              Fog     76334             11.7436     < 0.001     0.2700       small
             Mist    531924              8.8361     < 0.001     0.2040       small
          Thunder     93418             40.9606     < 0.001     0.9470       large
         Freezing     10715             40.1237     < 0.001     0.9240       large
